# IAM Handwriting TrOCR Validation
A script to validate TrOCR inference and sentence reconstruction on the IAM Handwriting Database.

In [ ]:
!pip install kaggle transformers torch pillow groq

In [ ]:
import os

# Use the new Kaggle API Token format
os.environ['KAGGLE_API_TOKEN'] = "KGAT_1673a594e9522d05936843f6013b272c"

print("Kaggle credentials configured. Downloading dataset...")
!kaggle datasets download -d nibinv23/iam-handwriting-word-database -p /content/data --unzip
print("Dataset downloaded and extracted to /content/data")

### Step 1: TrOCR Inference

In [ ]:
import os
import glob
import torch
from transformers import AutoProcessor, VisionEncoderDecoderModel, logging
from PIL import Image
import warnings

# Suppress warnings for clean output
logging.set_verbosity_error()
warnings.filterwarnings("ignore")

print("Loading TrOCR processor and model...")
processor = AutoProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

# Dynamically find the words directory in case Kaggle nested it inside another folder
words_dir = glob.glob("/content/data/**/words", recursive=True)
words_dir = words_dir[0] if words_dir else "/content/data/words"

# Sample images from the IAM dataset
image_paths = [
    os.path.join(words_dir, "a01/a01-000u/a01-000u-00-00.png"),
    os.path.join(words_dir, "a01/a01-000u/a01-000u-00-01.png"),
    os.path.join(words_dir, "a01/a01-000u/a01-000u-00-02.png"),
]

print("\nRunning inference on sample images...")
for path in image_paths:
    try:
        # Load and convert image to RGB as required by TrOCR
        image = Image.open(path).convert("RGB")
        
        # Preprocess the image
        pixel_values = processor(image, return_tensors="pt").pixel_values
        
        # Generate text (set max_new_tokens to avoid the warning)
        generated_ids = model.generate(pixel_values, max_new_tokens=21)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        print(f"Image: {path}\nPredicted Text: '{generated_text}'\n")
    except Exception as e:
        print(f"Error processing {path}: {e}")

### Step 2: Sentence Reconstruction and Accuracy Validation

In [ ]:
import os
import glob
import torch
from transformers import AutoProcessor, VisionEncoderDecoderModel, logging
from PIL import Image
import warnings

# Dynamically find words.txt and the words directory
words_file_matches = glob.glob("/content/data/**/words.txt", recursive=True)
if not words_file_matches:
    print("Could not find words.txt in the dataset folder! Make sure the Kaggle download cell ran successfully.")
else:
    words_file = words_file_matches[0]
    dataset_root = os.path.dirname(words_file)
    words_dir = os.path.join(dataset_root, "words")
    groups = {}

    print("Parsing words.txt...")
    # 1. Parse words.txt
    with open(words_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            
            parts = line.strip().split()
            if len(parts) >= 9:
                word_id = parts[0]
                status = parts[1]
                
                # Filter to rows where status == "ok" only
                if status != "ok":
                    continue
                
                transcription = " ".join(parts[8:]) 
                
                # wordID format: formID-lineNum-wordPos (e.g. a01-000u-00-00)
                id_parts = word_id.split('-')
                if len(id_parts) >= 4:
                    form_id = f"{id_parts[0]}-{id_parts[1]}"
                    line_num = id_parts[2]
                    word_pos = int(id_parts[3])
                    
                    # 2. Group entries by form+line prefix
                    line_prefix = f"{form_id}-{line_num}"
                    
                    if line_prefix not in groups:
                        groups[line_prefix] = []
                    
                    groups[line_prefix].append({
                        "word_id": word_id,
                        "form_id": form_id,
                        "word_pos": word_pos,
                        "transcription": transcription
                    })

    # Pick 3 different line groups to reconstruct
    selected_groups = []
    for prefix, words in groups.items():
        if len(words) >= 5: # Pick lines that have at least 5 words for a good sentence
            selected_groups.append(prefix)
            if len(selected_groups) == 3:
                break
                
    if len(selected_groups) < 3:
        print("Could not find 3 suitable line groups.")
    else:
        print("Loading TrOCR processor and model...")
        processor = AutoProcessor.from_pretrained("microsoft/trocr-base-handwritten")
        model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")
        
        reconstructed_sentences_for_groq = []
        # Process the 3 selected line groups
        for prefix in selected_groups:
            words = groups[prefix]
            
            # 3. Within each group, sort by word position number
            words.sort(key=lambda x: x['word_pos'])
            
            gt_words = []
            pred_words = []
            
            for w in words:
                gt_words.append(w['transcription'])
                
                # Construct image path using dynamic words_dir
                dir1 = w['form_id'].split('-')[0]
                dir2 = w['form_id']
                img_path = os.path.join(words_dir, dir1, dir2, w['word_id'] + ".png")
                
                try:
                    # 4. Load each word's image file and run TrOCR
                    image = Image.open(img_path).convert("RGB")
                    pixel_values = processor(image, return_tensors="pt").pixel_values
                    generated_ids = model.generate(pixel_values, max_new_tokens=21)
                    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
                    pred_words.append(generated_text)
                except Exception as e:
                    print(f"Error processing {img_path}: {e}")
                    pred_words.append("[ERROR]")
                    
            # 4 & 5. Join words to form reconstructed and reference sentences
            gt_sentence = " ".join(gt_words)
            pred_sentence = " ".join(pred_words)
            
            print(f"\nDiagnostic: len(gt_words)={len(gt_words)}, len(pred_words)={len(pred_words)}")
            if prefix == "a01-000u-01":
                print(f"Raw GT List  : {gt_words}")
                print(f"Raw Pred List: {pred_words}")
            
            # 6. Compute word-level accuracy (case-insensitive match count / total words)
            matches = 0
            for gw, pw in zip(gt_words, pred_words):
                # Evaluate using case-insensitive string matching
                if gw.lower() == pw.lower():
                    matches += 1
            accuracy = (matches / len(gt_words)) * 100 if gt_words else 0.0
            
            print(f"\n--- Line Group: {prefix} ---")
            print(f"Ground Truth : {gt_sentence}")
            print(f"Prediction   : {pred_sentence}")
            print(f"Accuracy     : {matches}/{len(gt_words)} ({accuracy:.1f}%)")
            reconstructed_sentences_for_groq.append(pred_sentence)

### Step 3: LLM Explanation via Groq

In [ ]:
import os
from groq import Groq

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception as e:
    print(f"Could not retrieve GROQ_API_KEY from Colab secrets: {e}")
    print("Please ensure you have added GROQ_API_KEY to the Secrets tab on the left sidebar.")
    import getpass
    GROQ_API_KEY = getpass.getpass("Alternatively, paste your Groq API Key here: ").strip()
    if not GROQ_API_KEY:
        raise ValueError("API Key cannot be empty. Please provide a valid Groq API Key.")

client = Groq(api_key=GROQ_API_KEY)

def explain_reconstruction(sentence):
    system_prompt = (
        "You are an AI assistant correcting OCR output from handwriting recognition. "
        "For each sentence you receive, respond with EXACTLY this format:\n\n"
        "Corrected: <your best corrected transcription>\n"
        "Confidence: <HIGH | MEDIUM | LOW>\n"
        "Note: <any notes, or omit this line if confidence is HIGH>\n"
        "Context: <1-2 sentence explanation, ONLY if confidence is HIGH>\n\n"
        "Rules:\n"
        "1. Fix obvious OCR/transcription errors (misspellings, garbled tokens) "
        "while preserving the original meaning.\n"
        "2. Assign a confidence label for the correction as a whole:\n"
        "   - HIGH: the corrected sentence clearly reflects the intended meaning.\n"
        "   - MEDIUM: some words are uncertain but the gist is likely correct.\n"
        "   - LOW: more than 2 words were substantially altered, or the corrected "
        "sentence's meaning is a guess rather than a clear fix.\n"
        "3. If confidence is MEDIUM or LOW, you MUST include the note: "
        "'This reconstruction may not reflect the original meaning.' "
        "Do NOT provide a contextual explanation paragraph in that case.\n"
        "4. Do NOT invent specific details (times, named actions, first-person narrative) "
        "that are not clearly present in the input tokens.\n"
        "5. Do not output any additional chat or conversational text."
    )
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"OCR output:\n{sentence}"}
        ],
        temperature=0.2,
        max_tokens=300
    )
    return response.choices[0].message.content

print("\n--- Step 3: LLM Explanation via Groq ---\n")
for i, sentence in enumerate(reconstructed_sentences_for_groq, 1):
    print(f"[Line Group {i}]")
    print(f"  OCR Input : {sentence}")
    print(f"  LLM Output:\n{explain_reconstruction(sentence)}\n")
    print("-" * 50)